# Import

In [ ]:
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import association_rules

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Import Dataset

In [ ]:
# 1000000 line of data
# Load csv file
df_ride = pd.read_csv("../data/RAW_DATA (2023_Yellow_Taxi_Trip_Data).csv", nrows=1000000)
df_zone = pd.read_csv("../data/RAW_DATA (taxi_zone_lookup).csv", nrows=1000000)

# Load excel file
df_weather = pd.read_excel("../data/RAW_DATA (new york weather 2023).xlsx").head(1000000)

# Data Understanding

In [ ]:
df_ride.info()
df_ride.isnull().sum()

In [ ]:
df_ride.describe()

In [ ]:
df_weather.info()
df_weather.isnull().sum()

In [ ]:
df_weather.describe()

In [ ]:
df_zone.info()
df_zone.isnull().sum()

In [ ]:
df_zone.describe()
# only the numerical data is described

# Data Preparation

In [ ]:
# Outliers detection IQR
# Trip distance
Q1_distance = df_ride['trip_distance'].quantile(0.25)
Q3_distance = df_ride['trip_distance'].quantile(0.75)
IQR_distance = Q3_distance - Q1_distance

df_ride = df_ride[((df_ride['trip_distance'] >= (Q1_distance - 1.5 * IQR_distance)) & (df_ride['trip_distance'] <= (Q3_distance + 1.5 * IQR_distance)))]

In [ ]:
# Remove the outliers and inconsistent data
df_ride = df_ride[((df_ride['passenger_count'] > 0) & (df_ride['fare_amount'] > 0.0) & (df_ride['trip_distance'] > 0.0))]
df_ride.head()

In [ ]:
# Fare amount
Q1_fare = df_ride['fare_amount'].quantile(0.25)
Q3_fare = df_ride['fare_amount'].quantile(0.75)
IQR_fare = Q3_fare - Q1_fare

df_ride = df_ride[((df_ride['fare_amount'] >= (Q1_fare - 1.5 * IQR_fare)) & (df_ride['fare_amount'] <= (Q3_fare + 1.5 * IQR_fare)))]

In [ ]:
# After removing outliers
df_ride.describe()

In [ ]:
# Check for inconsistent and noisy data
df_ride.min()

In [ ]:
# Remove the rows with missing values in the zone data
df_zone.dropna()

In [ ]:
# Remove the rows with missing values in the weather data
df_weather.dropna()

In [ ]:
# Check the column names of the data
df_ride.columns

In [ ]:
df_weather.columns

In [ ]:
df_zone.columns

In [ ]:
# Select the needed column for the analysis in weather data
columns_needed_weather = ['datetime', 'icon']
df_subset_weather = df_weather[columns_needed_weather]

In [ ]:
df_subset_weather.head()

In [ ]:
# Check the category of the weather condition
df_subset_weather['icon'].unique()
df_subset_weather['icon'].value_counts()

In [ ]:
# Select the needed column for the analysis in zone data
columns_needed_zone = ['LocationID', 'Borough']
df_subset_zone = df_zone[columns_needed_zone]

In [ ]:
df_subset_zone.head()

In [ ]:
# Check the category of the borough
df_subset_zone['Borough'].unique()
df_subset_zone['Borough'].value_counts()

In [ ]:
# Remove the rows with unknown and EWR in the borough data
df_subset_zone = df_subset_zone[((df_subset_zone['Borough'] != 'Unknown') & (df_subset_zone['Borough'] != 'EWR'))]
df_subset_zone.head()

In [ ]:
df_subset_zone['Borough'].unique()
df_subset_zone['Borough'].value_counts()

In [ ]:
# Select the needed column for the analysis
columns_needed_ride = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID','fare_amount', 'trip_distance', 'passenger_count']
df_subset_ride = df_ride[columns_needed_ride]

In [ ]:
# Number of duplicate of ride data
num_duplicates = df_subset_ride.duplicated().sum()
print("Number of duplicate rows:", num_duplicates)

# 7589 duplicates in 1000000 rows

In [ ]:
# Drop duplicates
df_subset_ride = df_subset_ride.drop_duplicates(subset=['tpep_pickup_datetime', 'tpep_dropoff_datetime'], keep='first')

In [ ]:
# Number of duplicate of weather data
num_duplicates = df_subset_weather.duplicated().sum()
print("Number of duplicate rows:", num_duplicates)

# No duplicate

In [ ]:
# Number of duplicate of zone data
num_duplicates = df_subset_zone.duplicated().sum()
print("Number of duplicate rows:", num_duplicates)

# No duplicate

In [ ]:
# Split the pickup and dropoff datetime column into date and time
df_subset_ride[['pickup_date', 'pickup_time', 'pickup_am_pm']] = df_subset_ride['tpep_pickup_datetime'].str.split(' ', expand=True)
df_subset_ride['pickup_time'] = df_subset_ride['pickup_time'] + ' ' + df_subset_ride['pickup_am_pm']
df_subset_ride = df_subset_ride.drop(columns=['pickup_am_pm'])

df_subset_ride[['dropoff_date', 'dropoff_time', 'dropoff_am_pm']] = df_subset_ride['tpep_dropoff_datetime'].str.split(' ', expand=True)
df_subset_ride['dropoff_time'] = df_subset_ride['dropoff_time'] + ' ' + df_subset_ride['dropoff_am_pm']
df_subset_ride = df_subset_ride.drop(columns=['dropoff_am_pm'])

In [ ]:
# drop pickup and dropoff datetime column
df_subset_ride = df_subset_ride.drop(columns=['tpep_pickup_datetime', 'tpep_dropoff_datetime'])

In [ ]:
# Change date format
df_subset_weather['datetime'] = pd.to_datetime(df_subset_weather['datetime'], errors='coerce')
df_subset_weather['datetime'] = df_subset_weather['datetime'].dt.strftime('%m/%d/%Y')

In [ ]:
df_subset_ride.head(15)

In [ ]:
df_subset_ride.describe()

In [ ]:
df_subset_weather.head(10)

In [ ]:
df_subset_zone.head(10)

# Final Joining

In [ ]:
# Join ride and zone
df_ride_and_zone = pd.merge(df_subset_ride, df_subset_zone, left_on='PULocationID', right_on='LocationID', how='left')

In [ ]:
df_ride_and_zone.head(15)

In [ ]:
# Join all datasets together
df_final = pd.merge(df_ride_and_zone, df_subset_weather, left_on='pickup_date', right_on='datetime', how='left')

In [ ]:
# Remove unused columns
df_final = df_final.drop(columns=['LocationID', 'datetime'])

In [ ]:
# Number of duplicate of weather data
num_duplicates = df_final.duplicated().sum()
print("Number of duplicate rows:", num_duplicates)

# No duplicate

In [ ]:
# Rename the columns
df_final = df_final.rename(columns={
    'pickup_date': 'pickup_date',
    'pickup_time': 'pickup_time',
    'PULocationID': 'pickup_location_id',
    'DOLocationID': 'dropoff_location_id',
    'fare_amount': 'fare_amount',
    'trip_distance': 'trip_distance',
    'passenger_count': 'passenger_count',
    'Borough': 'pickup_borough',
    'icon': 'weather'
})

# Cleaned data

In [ ]:
df_final.head()

# Assiciation Rule Mining

In [ ]:
# Categorized the pickup time
df_final['PickupTimeGroup'] = pd.to_datetime(df_final['pickup_time'], format="%I:%M:%S %p").dt.hour.map(
    lambda h: 'Night' if 0 <= h < 6 else
              'Morning' if 6 <= h < 12 else
              'Afternoon' if 12 <= h < 18 else
              'Evening'
)

In [ ]:
# Categorized the fare amount
df_final['FareGroup'] = pd.qcut(df_final['fare_amount'], 4, labels=['Low Price', 'Medium Price', 'High Price', 'Premium Price'])

In [ ]:
# Categorized the trip distance
df_final['TripDistanceGroup'] = pd.qcut(df_final['trip_distance'], 4, labels=['Short Distance', 'Medium Distance', 'Long Distance', 'Very Long Distance'])

In [ ]:
df_final.head()

In [ ]:
# Check the unique values of the new columns
df_final['PickupTimeGroup'].value_counts()

In [ ]:
df_final['FareGroup'].value_counts()

In [ ]:
df_final['TripDistanceGroup'].value_counts()

In [ ]:
df_final['weather'].value_counts()

In [ ]:
# Change the weather group data
df_final['WeatherGroup'] = df_final['weather'].replace({
    'rain': 'Bad Weather',
    'partly-cloudy-day': 'Good Weather'
})

In [ ]:
df_final['WeatherGroup'].value_counts()

In [ ]:
df_final.head()

In [ ]:
df_final.isnull().sum()

In [ ]:
# Remove the rows with missing values
df_final = df_final.dropna(subset=['pickup_borough'])
df_final = df_final.dropna(subset=['weather'])
df_final = df_final.dropna(subset=['WeatherGroup'])

In [ ]:
df_final.isnull().sum()

In [ ]:
# Association rules analysis
# Change data into list / Contatenate the data / make it transaction format
df_final['CombinedRule'] = df_final[['WeatherGroup', 'PickupTimeGroup', 'FareGroup', 'TripDistanceGroup']].values.tolist()


In [ ]:
df_final.head()

In [ ]:
# Change the data type
df_final['PickupTimeGroup'] = df_final['PickupTimeGroup'].astype('category')
df_final['WeatherGroup'] = df_final['WeatherGroup'].astype('category')

In [ ]:
df_final.info()

In [ ]:
df_final.describe()

In [ ]:
# one-hot encoding
te = TransactionEncoder()
te_array = te.fit(df_final['CombinedRule']).transform(df_final['CombinedRule'])

df_encoded = pd.DataFrame(te_array, columns=te.columns_)

In [ ]:
# Generate frequent itemsets with a minimum support of 0.05
frequent_itemsets = apriori(df_encoded, min_support=0.05, use_colnames=True)

In [ ]:
# Generate association rules with a minimum confidence of 0.6
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)

In [ ]:
# frozen set to reable format
rules['antecedents'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))

In [ ]:
# filter by lift and confidence in descending order
rules1 = rules.sort_values(by='lift', ascending=False)
rules2 = rules.sort_values(by='confidence', ascending=False)

In [ ]:
rules1.head(5)

In [ ]:
rules2.head(5)

# Descriptive Analysis

In [ ]:
# Number of rides by borough
zone = df_final.groupby(by='pickup_borough')['pickup_location_id'].size()
plt.figure(figsize=(12,6))
plt.bar(x=zone.index, height=zone.values, color='yellow', alpha=0.3)

# Data labels
for borough, count in zip(zone.index, zone.values):
    plt.text(borough, count + 0.5, str(count), ha='center', va='bottom', color='yellow')

plt.xlabel('Borough')
plt.ylabel('Number of Rides')
plt.title('Number of Rides in Each Borough')
plt.tight_layout()
plt.savefig("../images/Number_of_Rides_by_Borough.png")
plt.show()

In [ ]:
# Calculate the number of rides by weather condition
weather_icon = df_final.groupby('WeatherGroup')['pickup_location_id'].size()
plt.figure(figsize=(12, 6))
plt.bar(x=weather_icon.index, height=weather_icon.values, color='green', alpha=0.3)

# Data labels
for x, y in zip(weather_icon.index, weather_icon.values):
    plt.text(x, y + 2, str(y), ha='center', va='bottom', color='green')


plt.xlabel('Weather Condition')
plt.ylabel('Number of Rides')
plt.title('Number of Rides for Each Weather Condition')
plt.tight_layout()
plt.savefig("../images/Number_of_Rides_by_Weather_Group.png")
plt.show()

In [ ]:
# Number of rides by pickup time group
pickup_counts = df_final.groupby('PickupTimeGroup')['pickup_location_id'].count()

color = ['red','orange','yellow','pink']

plt.figure(figsize=(6,6))
plt.pie(pickup_counts, 
        labels=pickup_counts.index, 
        autopct='%1.1f%%', 
        startangle=90, 
        colors=color, wedgeprops={'alpha': 0.5})
plt.title('Proportion of Rides by Pickup Time Group')
plt.savefig("../images/Number_of_Rides_by_Pickup_Time_Group.png")
plt.show()

In [ ]:
# Number of rides by fare group
fare = df_final.groupby('FareGroup')['pickup_location_id'].size()

plt.figure(figsize=(12, 6))
plt.bar(fare.index, fare.values, color='blue', alpha=0.3)

# Data labels
for fare_group, count in zip(fare.index, fare.values):
	plt.text(fare_group, count + 2, str(count), ha='center', va='bottom', color='blue')

plt.xlabel('Fare Group')
plt.ylabel('Number of Rides')
plt.title('Number of Rides for Each Fare Group')
plt.tight_layout()
plt.savefig("../images/Number_of_Rides_by_Fare_Group.png")
plt.show()

In [ ]:
counts = df_final.pivot_table(index='PickupTimeGroup', columns='TripDistanceGroup', 
                              values='pickup_location_id', aggfunc='count', fill_value=0)

counts.plot(kind='bar', figsize=(12,6), alpha=0.8)
plt.xlabel('Pickup Time Group')
plt.ylabel('Number of Rides')
plt.title('Number of Rides by Pickup Time and Trip Distance Group')
plt.xticks(rotation=0)
plt.legend(title='Trip Distance Group')
plt.savefig("../images/Number_of_Rides_by_Pickup_Time_and_Trip_Distance_Group.png")
plt.show()

# Random Forest

In [ ]:
df_final.columns

In [ ]:
# x is factors, y is the result that want to predict
X = df_final[['trip_distance', 'WeatherGroup', 'PickupTimeGroup', 'pickup_borough']]
y = df_final['fare_amount']

In [ ]:
# Always remeber do one-hot encoding before split the data!!
X = pd.get_dummies(X, drop_first=True)

In [ ]:
# Split the data into training and testing sets with 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# 100 trees
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
# MAE is for accuracy and RMSE is detect the big error
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

In [ ]:
# Most influnecial factors
importance = pd.Series(model.feature_importances_, index=X.columns)
importance.sort_values(ascending=False)

# Model Evaluation and Visualization

In [ ]:
# Most influnecial factors visualization
importance.sort_values().plot(kind='barh', figsize=(12, 6), color='purple', alpha=0.2)
plt.title("Feature Importance (Random Forest)")
plt.savefig("../images/Feature_Importance(Random_Forest).png")
plt.show()

In [ ]:
# Actual vs Predicted Fare
plt.figure(figsize=(12,6))
plt.scatter(y_test, y_pred, alpha=0.3, color='indigo')
plt.xlabel("Actual Fare")
plt.ylabel("Predicted Fare")
plt.title("Actual vs Predicted Fare")
plt.savefig("../images/Actual_vs_Predicted_Fare.png")
plt.show()

In [ ]:
# Prediction error distribution
errors = y_test - y_pred

plt.figure(figsize=(12, 6))
plt.hist(errors, bins=30, color='orange', alpha=0.5)
plt.title("Prediction Error Distribution")
plt.xlabel("Prediction Error")
plt.ylabel("Frequency")
plt.savefig("../images/Prediction_Error_Distribution.png")
plt.show()